<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week2_cleaning_text_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 2 — Cleaning Text (STARTER)
### The NLP Internship | LinguaAI Labs

This week: build a text cleaning pipeline, apply it to 50,000 tickets, and save `tickets_clean.csv`.

**Fill in every `# YOUR CODE HERE` block.**

In [ ]:
import sys, subprocess, importlib, re, unicodedata
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
for pkg,imp in [("spacy","spacy"),("nltk","nltk"),("langdetect","langdetect")]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
import spacy, nltk
from langdetect import detect, LangDetectException
from collections import Counter
for r in ["punkt","punkt_tab","stopwords","wordnet"]:
    nltk.download(r, quiet=True)
try: nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")
np.random.seed(42)
print("Setup complete ✅")

## Part 1 — Noise Audit

> ⏸️ **Pause and Predict**
> Before running: which noise type do you expect to be most frequent — emoji, ALL CAPS, or repeated punctuation?

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/support_tickets.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/nlp-internship-in-a-book/main/data/support_tickets.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload support_tickets.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        tickets = [
            'My account login is not working, please help',
            'I was charged twice for my subscription this month',
            'The app crashes whenever I try to upload a file',
            'How do I cancel my subscription?',
            'Great service, very happy with the product!',
            'I cannot reset my password, the email never arrives',
            'The dashboard is loading very slowly today',
            'I need a refund for my last order',
            'Feature request: can you add dark mode?',
            'Error 500 when trying to export my data',
        ] * 50
        np.random.shuffle(tickets)
        df = pd.DataFrame({
            'ticket_id':  range(1, 501),
            'text':       tickets[:500],
            'category':   np.random.choice(['billing','technical','account','general'], 500),
            'sentiment':  np.random.choice(['positive','negative','neutral'], 500, p=[0.3,0.4,0.3]),
            'priority':   np.random.choice(['low','medium','high'], 500),
        })
        print('Sample dataset ready (500 tickets) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## Part 2 — Build the Cleaning Functions

In [ ]:
# YOUR CODE HERE — implement each function
# Reference: Week 2 chapter for full specifications

EMOJI_MAP = {
    "😭":" distressed ","😤":" angry ","🙏":" pleading ",
    "😡":" very_angry ","😢":" sad ","🤦":" frustrated ",
}
PIDGIN_MARKERS = {"abeg","una","don","dey","wahala","wetin","shey","abi","sha"}

def map_emoji(text):
    # Replace top emoji with labels; remove all remaining non-ASCII
    # YOUR CODE HERE
    pass

def remove_urls(text):
    # YOUR CODE HERE
    pass

def remove_pii(text):
    # Replace phone numbers (7+ digits) with PHONE_REDACTED
    # Replace email addresses with EMAIL_REDACTED
    # YOUR CODE HERE
    pass

def reduce_repeated(text, max_repeat=3):
    # "soooooo" → "sooo", "!!!" → "!!!"
    # YOUR CODE HERE
    pass

def normalise_whitespace(text):
    # YOUR CODE HERE
    pass

def clean_ticket(text):
    """Full cleaning pipeline — apply in order."""
    if not isinstance(text, str): return ""
    text = map_emoji(text)
    text = remove_urls(text)
    text = remove_pii(text)
    text = reduce_repeated(text)
    text = normalise_whitespace(text)
    return text

def detect_language(text):
    """Detect dominant language. Returns: en, yo, ha, pcm, other, unknown."""
    # YOUR CODE HERE (use langdetect.detect)
    pass

def is_code_switched(text):
    """Return True if text contains both Pidgin markers and English function words."""
    # YOUR CODE HERE
    pass

# Test your pipeline
test_tickets = [
    "URGENT!!! My account has been locked for 3 days!!! Please help 😭😭😭",
    "Abeg o, I don call una support line 08012345678 but nobody answer me 😤",
]
for raw in test_tickets:
    print(f"RAW:     {raw[:80]}")
    print(f"CLEANED: {clean_ticket(raw)[:80]}")
    print()

## Part 3 — Language Detection

In [ ]:
# Test language detection on sample tickets
samples = {
    "English": "I have been waiting three days for a response to my complaint.",
    "Pidgin":  "Abeg I don try reach una many times but nobody reply my message.",
    "Mixed":   "Please help me o, the app don crash again and my money don disappear.",
}
print("Language detection results:\n")
for label, text in samples.items():
    lang = detect_language(text)
    switched = is_code_switched(text)
    print(f"  [{label}] detected={lang}, code-switched={switched}")
    print(f"    Text: {text[:60]}")
    print()

## Part 4 — Apply at Scale

> 🧠 **What Will This Output?**
> Before running: predict the mean word count before vs after cleaning. Will cleaning reduce it significantly?

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

print("Applying cleaning pipeline to 50,000 tickets...")
df["text_clean"]       = df["text"].fillna("").progress_apply(clean_ticket)
df["lang_detected_v2"] = df["text_clean"].progress_apply(detect_language)
df["is_code_switched"] = df["text_clean"].progress_apply(is_code_switched)
df["word_count_clean"] = df["text_clean"].str.split().str.len()

print(f"\nCleaning complete.")
print(f"  Empty tickets after cleaning: {(df['text_clean'] == '').sum()}")
print(f"  Mean word count before: {df['text'].str.split().str.len().mean():.1f}")
print(f"  Mean word count after:  {df['word_count_clean'].mean():.1f}")

df.to_csv("datasets/tickets_clean.csv", index=False)
print("\nSaved: datasets/tickets_clean.csv ✅")
print("\n💡 Save clean_ticket(), detect_language(), is_code_switched() to clean_ticket.py")
print("   From Week 3 onward: from clean_ticket import clean_ticket, detect_language")

## Cleaning Log

> Document every decision in a markdown cell — what you cleaned, why, and what you risked.

| Decision | What | Why | Risk |
|----------|------|-----|------|
| Emoji | Map top-10 → labels; remove rest | Emotional signal | Emoji labels are subjective |
| URLs | Remove | No topical signal | Domain names lost |
| Phone/email | Redact | PII compliance | None |
| Repeated chars | Reduce to 3 | Preserve emphasis | Degree of emphasis lost |
| Case | NOT lowercased | Acronyms carry meaning | Larger vocabulary |
| Stopwords | NOT removed | Sentiment signal in 'not', 'cannot' | TF-IDF noisier |

*Add more rows for any additional decisions you made.*

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Removing all punctuation blindly | Punctuation carries meaning. 'Not bad' and 'Not. Bad.' are different. Question marks signal uncertainty. Exclamation marks signal intensity. | Decide per-punctuation-type whether to keep or remove |
| Cleaning training and test data separately | If you normalise the training set differently from the test set, your model sees different distributions. Always fit your cleaner on training data only. | Fit on train, transform train and test with the same fitted cleaner |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Write a reusable `clean_text(text)` function that applies all your cleaning decisions in the correct order. It should handle None values without crashing.

> 🔬 **Advanced Path**
> Apply your cleaning function to the full dataset and compare vocabulary size before and after. How many unique tokens did you remove? Which ones surprised you?

## ✅ What You Can Do After This Week

- Audit a corpus systematically for noise types
- Build a modular text cleaning pipeline
- Apply language-aware cleaning rules
- Save `tickets_clean.csv` and the `clean_ticket.py` module

---
## ✅ Week 2 Complete
**Next:** `week3_finding_topics_STARTER.ipynb`

---
*Add `week2_cleaning_text_STARTER.ipynb` to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 2, you can now:

- Implement a reproducible text cleaning pipeline with all decisions documented
- Explain why cleaning choices must be matched to the downstream task
- Fit a text cleaner on training data only — never on the full dataset
- Write a cleaning log a future team member could follow without asking questions

📤 **GitHub:** Push week2_cleaning_text_STARTER.ipynb and cleaning_decisions.md. Commit: "Week 2: Text cleaning — all decisions documented"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
